# FinBERT Sentiment Analysis - MIAX14 Hackathon

Usamos [ProsusAI/finbert](https://huggingface.co/ProsusAI/finbert) para extraer sentimiento financiero
de los titulares de noticias. FinBERT produce 3 clases: **positive / negative / neutral**.

**Output**: `data/finbert_sentiment_train.csv` y `data/finbert_sentiment_test.csv` con features
diarias listas para usar en el pipeline de entrenamiento.

In [ ]:
# Instalar dependencias si no están
import subprocess, sys
def pip(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

pip('transformers')
pip('torch')
pip('tqdm')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 4)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Cargar FinBERT

In [ ]:
MODEL_NAME = 'ProsusAI/finbert'
print(f'Cargando {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()
model.to(DEVICE)

# Labels: 0=positive, 1=negative, 2=neutral (orden del modelo original)
LABEL_NAMES = model.config.id2label
print('Labels:', LABEL_NAMES)

## 2. Función de inferencia en batch

In [ ]:
@torch.no_grad()
def score_batch(texts: list[str], batch_size: int = 64) -> pd.DataFrame:
    """
    Returns a DataFrame with columns [positive, negative, neutral, sentiment_score]
    for each input text. sentiment_score = positive - negative (range -1 to +1).
    """
    all_probs = []
    for i in tqdm(range(0, len(texts), batch_size), desc='FinBERT inference'):
        batch = texts[i:i + batch_size]
        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors='pt',
        ).to(DEVICE)
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

    probs_all = np.vstack(all_probs)  # (N, 3)

    # Map label ids to names
    col_order = [LABEL_NAMES[i] for i in range(3)]
    df = pd.DataFrame(probs_all, columns=col_order)

    # Ensure positive/negative/neutral exist regardless of label order
    for col in ['positive', 'negative', 'neutral']:
        if col not in df.columns:
            df[col] = 0.0

    df['sentiment_score'] = df['positive'] - df['negative']
    return df

## 3. Cargar noticias y puntuar

In [ ]:
train_news = pd.read_csv('../data/train_news.csv', parse_dates=['Date'])
test_news  = pd.read_csv('../data/test_news.csv',  parse_dates=['Date'])

print(f'Train noticias: {len(train_news):,}')
print(f'Test noticias:  {len(test_news):,}')
train_news.head(3)

In [ ]:
# FinBERT en train
train_headlines = train_news['Headline'].fillna('').tolist()
train_scores = score_batch(train_headlines)
train_news_scored = pd.concat([train_news.reset_index(drop=True), train_scores], axis=1)
train_news_scored.head()

In [ ]:
# FinBERT en test
test_headlines = test_news['Headline'].fillna('').tolist()
test_scores = score_batch(test_headlines)
test_news_scored = pd.concat([test_news.reset_index(drop=True), test_scores], axis=1)
test_news_scored.head()

## 4. Agregación diaria de sentimiento

In [ ]:
def aggregate_daily(df: pd.DataFrame) -> pd.DataFrame:
    """Agrega scores de FinBERT a nivel diario."""
    agg = df.groupby('Date').agg(
        finbert_count        = ('sentiment_score', 'count'),
        finbert_sentiment    = ('sentiment_score', 'mean'),
        finbert_sentiment_std= ('sentiment_score', 'std'),
        finbert_positive     = ('positive', 'mean'),
        finbert_negative     = ('negative', 'mean'),
        finbert_neutral      = ('neutral', 'mean'),
        finbert_bull_ratio   = ('positive', lambda x: (x > 0.6).mean()),  # fracción muy positivas
        finbert_bear_ratio   = ('negative', lambda x: (x > 0.6).mean()),  # fracción muy negativas
    ).fillna(0)
    return agg

train_daily = aggregate_daily(train_news_scored)
test_daily  = aggregate_daily(test_news_scored)

print('Train daily shape:', train_daily.shape)
train_daily.describe().round(4)

## 5. Distribución de sentimiento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train_news_scored['sentiment_score'].hist(bins=50, ax=axes[0])
axes[0].set_title('Distribución de sentiment_score (por titular)')

train_daily['finbert_sentiment'].plot(ax=axes[1], lw=0.7)
axes[1].set_title('Sentimiento medio diario (train)')

train_daily['finbert_count'].plot(ax=axes[2], lw=0.7, color='green')
axes[2].set_title('Noticias por día (train)')

plt.tight_layout()
plt.show()

## 6. Correlación de sentimiento con retornos de índices

In [ ]:
idx = pd.read_csv('../data/train_indices.csv', parse_dates=['Date'], index_col='Date')
returns = idx.pct_change()

sent_cols = ['finbert_sentiment', 'finbert_positive', 'finbert_negative', 'finbert_bull_ratio', 'finbert_bear_ratio']
combined = returns.join(train_daily[sent_cols], how='inner')

corr = combined.corr().loc[sent_cols, returns.columns]

plt.figure(figsize=(10, 4))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0)
plt.title('Correlación FinBERT sentiment ↔ retornos diarios')
plt.tight_layout()
plt.show()

## 7. Análisis por ticker (¿qué índice mueve cada empresa?)

In [ ]:
# Sentimiento medio por ticker y su correlación con retornos
ticker_agg = (
    train_news_scored
    .groupby(['Date', 'Ticker_Anonymized'])['sentiment_score']
    .mean()
    .unstack(fill_value=0)
)

print(f'Tickers únicos: {ticker_agg.shape[1]}')

# Correlación de cada ticker con cada índice
corr_ticker = ticker_agg.join(returns, how='inner').corr()
corr_ticker_idx = corr_ticker.loc[ticker_agg.columns, returns.columns]

# Top 10 tickers más correlacionados con cada índice
for index_name in returns.columns:
    top = corr_ticker_idx[index_name].abs().sort_values(ascending=False).head(5)
    print(f'\nTop tickers para {index_name}:')
    print(top.to_string())

## 8. Features por índice (sentimiento de tickers relevantes)

In [ ]:
def build_index_specific_sentiment(
    news_df: pd.DataFrame,
    scores_df: pd.DataFrame,
    top_n_tickers: int = 5,
    corr_threshold: float = 0.05,
) -> pd.DataFrame:
    """
    Para cada índice, crea una feature de sentimiento basada en los tickers
    más correlacionados con ese índice.
    """
    scored = pd.concat([news_df.reset_index(drop=True), scores_df], axis=1)
    ticker_daily = (
        scored
        .groupby(['Date', 'Ticker_Anonymized'])['sentiment_score']
        .mean()
        .unstack(fill_value=np.nan)
    )

    combined = ticker_daily.join(returns, how='inner')

    result = {}
    for index_name in returns.columns:
        corrs = combined.corr()[index_name]
        top_tickers = (
            corrs
            .reindex(ticker_daily.columns)
            .dropna()
            .abs()
            .sort_values(ascending=False)
            .head(top_n_tickers)
            .index.tolist()
        )
        if top_tickers:
            col_name = f'finbert_{index_name}_ticker_sent'
            result[col_name] = ticker_daily[top_tickers].mean(axis=1)

    return pd.DataFrame(result)


idx_sent_train = build_index_specific_sentiment(train_news, train_scores)
print('Index-specific sentiment features shape:', idx_sent_train.shape)
idx_sent_train.head()

## 9. Guardar features finales

In [ ]:
# Combinar features diarias + por índice (train)
train_final = train_daily.join(idx_sent_train, how='left').fillna(0)
train_final.index.name = 'Date'
train_final.to_csv('../data/finbert_sentiment_train.csv')
print('Guardado: data/finbert_sentiment_train.csv')
print(train_final.shape)
train_final.tail()

In [ ]:
# Test: construir las mismas features por ticker usando los top tickers identificados en train
test_news_scored_copy = pd.concat([test_news.reset_index(drop=True), test_scores], axis=1)

ticker_daily_train = (
    train_news_scored
    .groupby(['Date', 'Ticker_Anonymized'])['sentiment_score']
    .mean()
    .unstack(fill_value=np.nan)
)
combined_train = ticker_daily_train.join(returns, how='inner')

ticker_daily_test = (
    test_news_scored_copy
    .groupby(['Date', 'Ticker_Anonymized'])['sentiment_score']
    .mean()
    .unstack(fill_value=np.nan)
)

idx_sent_test_dict = {}
for index_name in returns.columns:
    corrs = combined_train.corr()[index_name]
    top_tickers = (
        corrs
        .reindex(ticker_daily_train.columns)
        .dropna()
        .abs()
        .sort_values(ascending=False)
        .head(5)
        .index.tolist()
    )
    available = [t for t in top_tickers if t in ticker_daily_test.columns]
    if available:
        col_name = f'finbert_{index_name}_ticker_sent'
        idx_sent_test_dict[col_name] = ticker_daily_test[available].mean(axis=1)

idx_sent_test = pd.DataFrame(idx_sent_test_dict)
test_final = test_daily.join(idx_sent_test, how='left').fillna(0)
test_final.index.name = 'Date'
test_final.to_csv('../data/finbert_sentiment_test.csv')
print('Guardado: data/finbert_sentiment_test.csv')
test_final.head()

## 10. Visualización: sentimiento FinBERT vs retornos

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
for ax, col in zip(axes.flatten(), returns.columns):
    sent_key = f'finbert_{col}_ticker_sent'
    if sent_key in train_final.columns:
        combined_plot = returns[col].to_frame().join(train_final[[sent_key]], how='inner')
        ax2 = ax.twinx()
        combined_plot[col].rolling(21).mean().plot(ax=ax, color='steelblue', lw=0.9, label='Retorno 21d MA')
        combined_plot[sent_key].rolling(21).mean().plot(ax=ax2, color='orange', lw=0.9, alpha=0.8, label='FinBERT sent 21d MA')
        ax.set_title(col)
        ax.set_ylabel('Retorno', color='steelblue')
        ax2.set_ylabel('Sentimiento', color='orange')
    else:
        returns[col].rolling(21).mean().plot(ax=ax, lw=0.9)
        ax.set_title(f'{col} (sin ticker sent específico)')
plt.tight_layout()
plt.show()

## 11. Cómo integrar en el pipeline principal

Los archivos `data/finbert_sentiment_train.csv` y `data/finbert_sentiment_test.csv` ya están guardados.
Para usarlos en `src/features.py`, añade este bloque en `build_feature_matrix()`:

```python
# En load_data():
finbert_train = pd.read_csv('data/finbert_sentiment_train.csv', parse_dates=['Date'], index_col='Date')
finbert_test  = pd.read_csv('data/finbert_sentiment_test.csv',  parse_dates=['Date'], index_col='Date')

# En build_feature_matrix() al construir feat:
feat = feat.join(finbert_train, how='left').fillna(0)
test_feat = test_feat.join(finbert_test, how='left').fillna(0)
```

## 12. Generar predicción final con FinBERT

In [ ]:
import os, sys
sys.path.insert(0, '../src')

from features import load_data, build_feature_matrix, INDICES
from models import EnsembleModel, LGBM_DEFAULTS, XGB_DEFAULTS
from predict import predict_autoregressive
import lightgbm as lgb
import xgboost as xgb_lib
import pandas as pd
import numpy as np

INDEX_F_ACTIVE_DATE = pd.Timestamp('2020-03-10')
VAL_SIZE = 252

# Cargar datos — los CSVs de FinBERT del paso 9 se cargan automáticamente
print('Cargando datos...')
data = load_data('../data')
train_feat, test_feat = build_feature_matrix(data, data_dir='../data')

feature_cols = [c for c in train_feat.columns if c not in INDICES]
X = train_feat[feature_cols]
y = train_feat[INDICES]

X_train, X_val = X.iloc[:-VAL_SIZE], X.iloc[-VAL_SIZE:]
y_train, y_val = y.iloc[:-VAL_SIZE], y.iloc[-VAL_SIZE:]
X_train_f = X_train[X_train.index >= INDEX_F_ACTIVE_DATE]
y_train_f = y_train[y_train.index >= INDEX_F_ACTIVE_DATE]

print(f'Train : {len(X_train)} filas  ({X_train.index.min().date()} → {X_train.index.max().date()})')
print(f'Val   : {len(X_val)} filas   ({X_val.index.min().date()} → {X_val.index.max().date()})')
print(f'Features: {len(feature_cols)}')

In [ ]:
# Entrenar ensemble LightGBM + XGBoost
print('Entrenando ensemble...')
model_pred = EnsembleModel(INDICES, index_d_ghost_weight=0.7, lgbm_weight=0.5)

indices_full = [i for i in INDICES if i != 'Index_F']
model_pred.lgbm.indices = indices_full
model_pred.xgb.indices  = indices_full
model_pred.lgbm.fit(X_train, y_train, X_val, y_val)
model_pred.xgb.fit(X_train, y_train, X_val, y_val)

print('  [LGBM] Index_F (post-2020)...')
lgbm_f = lgb.LGBMRegressor(**LGBM_DEFAULTS)
lgbm_f.fit(X_train_f, y_train_f['Index_F'],
           eval_set=[(X_val, y_val['Index_F'])],
           callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(200)])
model_pred.lgbm.models['Index_F'] = lgbm_f

print('  [XGB]  Index_F (post-2020)...')
from xgboost import XGBRegressor
xgb_f = XGBRegressor(**(XGB_DEFAULTS | {'early_stopping_rounds': 100}))
xgb_f.fit(X_train_f, y_train_f['Index_F'],
          eval_set=[(X_val, y_val['Index_F'])], verbose=False)
model_pred.xgb.models['Index_F'] = xgb_f

for m in (model_pred.lgbm, model_pred.xgb):
    m.indices     = INDICES
    m.feature_cols = feature_cols
model_pred.feature_cols = feature_cols

# Validación rápida
val_pred = model_pred.predict(X_val, index_a_lag1=X_val.get('Index_A_lag1'))
print(f'\nRMSE validación (promedio): {float(np.sqrt(((y_val.values - val_pred.values)**2).mean())):.2f}')
for col in INDICES:
    r = float(np.sqrt(((y_val[col] - val_pred[col])**2).mean()))
    print(f'  {col:12s}: {r:>10.2f}')

In [ ]:
# Predicción autorregresiva y guardado en predicciones/predicciones_finBERT.xls
print('Generando predicciones de test (autorregresivo)...')
test_preds = predict_autoregressive(model_pred, data['train_idx'], test_feat)

out = test_preds.copy()
out.index.name = 'Date'
out = out.reset_index()
out['Date'] = pd.to_datetime(out['Date']).dt.strftime('%Y-%m-%d')

output_dir  = '../predicciones'
output_path = os.path.join(output_dir, 'predicciones_finBERT.xls')
os.makedirs(output_dir, exist_ok=True)

try:
    import xlwt  # formato .xls nativo
    out.to_excel(output_path, index=False, engine='xlwt')
except ImportError:
    # xlwt no disponible en Python ≥ 3.9: guardar .xlsx y renombrar a .xls
    import shutil
    tmp = output_path + 'x'          # .xlsx temporal
    out.to_excel(tmp, index=False)
    shutil.move(tmp, output_path)

print(f'\nArchivo guardado en: {os.path.abspath(output_path)}')
print(f'Filas: {len(out)}  |  Columnas: {list(out.columns)}')
out

## 13. Predicciones vs histórico

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# ── Cargar datos ─────────────────────────────────────────────────────────
hist = pd.read_csv('../data/train_indices.csv', parse_dates=['Date'], index_col='Date')

sub  = pd.read_excel('../predicciones/submission1_finBERT.xlsx', parse_dates=['Date'])
sub  = sub.set_index('Date')
# Renombrar pred_index_a → Index_A etc. para coincidir con hist
sub.columns = [c.replace('pred_index_', 'Index_').upper()
                .replace('INDEX_', 'Index_') for c in sub.columns]

INDICES = ['Index_A', 'Index_B', 'Index_C', 'Index_D', 'Index_E', 'Index_F']
HISTORY_DAYS = 500   # últimos N días de histórico mostrados

# ── Plot ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.suptitle('Predicciones FinBERT (naranja) vs Histórico (negro)', fontsize=15, y=1.01)

for ax, col in zip(axes.flatten(), INDICES):
    h = hist[col].iloc[-HISTORY_DAYS:]
    p = sub[col]

    # Punto de conexión: último valor histórico + primer día de predicción
    bridge_dates  = [h.index[-1], p.index[0]]
    bridge_values = [h.iloc[-1],  p.iloc[0]]

    ax.plot(h.index, h.values, color='black', lw=0.9, label='Histórico')
    ax.plot(bridge_dates, bridge_values, color='darkorange', lw=0.9)
    ax.plot(p.index, p.values, color='darkorange', lw=1.2, label='Predicción')

    # Línea vertical en el inicio de la predicción
    ax.axvline(x=p.index[0], color='gray', linestyle='--', lw=0.8, alpha=0.7)

    ax.set_title(col, fontsize=12)
    ax.set_xlabel('')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.savefig('../predicciones/predicciones_finBERT.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en predicciones/predicciones_finBERT.png')